# SynBioHub

![SynBioHub](https://proto-bio.github.io/proto-assets/images/tool/synbiohub/hero.png)

This notebook demonstrates `run_synbiohub_fetch`, which downloads the SBOL, FASTA, and GenBank representations of one design from a SynBioHub registry. SBOL keeps the design's composition (components, subcomponents, and annotations), while FASTA and GenBank are flattened sequence views. The running example is the iGEM device `BBa_I13504`, a GFP expression cassette built from an RBS, the GFP coding sequence, and a double terminator.

The public SynBioHub instance requires a login for downloads, including public records. Set `SYNBIOHUB_TOKEN` before running; the toolkit README's Toolkit Notes give the commands to create one.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("synbiohub")
display_overview("synbiohub")
display_docs_section("synbiohub", "Background")

# SynBioHub

The `synbiohub-fetch` tool retrieves exact SBOL, FASTA, and GenBank representations of a top-level design object from a caller-selected SynBioHub registry. It maintains no local parts catalog.

[SynBioHub](https://synbiohub.org/) is a standards-enabled repository for synthetic biology designs. SBOL preserves components, nested subcomponents, sequence constraints, annotations, and provenance, allowing a circuit or construct to remain a composition of segments rather than only a flattened nucleotide string. The tool uses the documented download endpoints and supports the optional `SYNBIOHUB_TOKEN` header; individual deployments may require login even for records displayed as public.

## Available tools

In [2]:
display_available_tools("synbiohub")

- **`run_synbiohub_fetch()`** — Fetch SBOL, FASTA, and GenBank representations from a SynBioHub registry

### `run_synbiohub_fetch`

`run_synbiohub_fetch` takes the URI of one top-level object and returns one document per requested format, each with the URL it was downloaded from. The object must be on the same host as `registry_url`, so a URI cannot redirect retrieval to another server. The token is read from `SYNBIOHUB_TOKEN` and never appears in cache keys or outputs.

In [3]:
import os

from proto_tools import (
    SynBioHubFetchConfig,
    SynBioHubFetchInput,
    run_synbiohub_fetch,
)

assert os.environ.get("SYNBIOHUB_TOKEN"), "set SYNBIOHUB_TOKEN before running this notebook"

In [4]:
# Display docs
display_api_reference("synbiohub", "input", "run_synbiohub_fetch")

# One top-level object on the public registry
inputs = SynBioHubFetchInput(object_uri="https://api.synbiohub.org/public/igem/BBa_I13504/1")

**Input** — `SynBioHubFetchInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>object_uri</code> | <code>str</code> | required | Absolute URI of a top-level object on the selected SynBioHub registry |

In [5]:
# Display config docs
display_api_reference("synbiohub", "config", "run_synbiohub_fetch")

# All three formats; recursive SBOL is the default | see docs above for all fields
config = SynBioHubFetchConfig(formats=["sbol", "fasta", "genbank"])

**Config** — `SynBioHubFetchConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>registry_url</code> | <code>str</code> | <code>'https://api.synbiohub.org'</code> | SynBioHub registry origin allowed for object retrieval |
| <code>formats</code> | <code>list[Literal['sbol', 'fasta', 'genbank']]</code> | <code>['sbol', 'fasta']</code> | Standards-native representations to retrieve |
| <code>recursive_sbol</code> | <code>bool</code> | <code>True</code> | Include referenced child objects in the SBOL document |
| <code>synbiohub_token</code> | <code>str &#124; None</code> | <code>$SYNBIOHUB_TOKEN</code> | Optional private-object token. Falls back to $SYNBIOHUB_TOKEN. |

In [6]:
# Run the fetch
result = run_synbiohub_fetch(inputs, config)

In [7]:
# Display output docs
display_api_reference("synbiohub", "output", "run_synbiohub_fetch")

print(f"{result.object_uri} from {result.registry_url}")
for document in result.documents:
    print(f"  {document.format:<8} {len(document.content):>7,} chars  {document.source_url}")

**Output** — `SynBioHubFetchOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>object_uri</code> | <code>str</code> | required | Canonical top-level object URI |
| <code>registry_url</code> | <code>str</code> | required | Registry origin used for retrieval |
| <code>documents</code> | <code>list[SynBioHubDocument]</code> | <code>[]</code> | Downloaded SBOL and sequence representations |

**`SynBioHubDocument`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>format</code> | <code>Literal['sbol', 'fasta', 'genbank']</code> | required | Downloaded representation format |
| <code>content</code> | <code>str</code> | required | Decoded document content |
| <code>source_url</code> | <code>str</code> | required | Stable SynBioHub download endpoint |

https://api.synbiohub.org/public/igem/BBa_I13504/1 from https://api.synbiohub.org
  sbol      38,793 chars  https://api.synbiohub.org/public/igem/BBa_I13504/1/sbol
  fasta        919 chars  https://api.synbiohub.org/public/igem/BBa_I13504/1/fasta
  genbank    2,133 chars  https://api.synbiohub.org/public/igem/BBa_I13504/1/gb


#### Read the Composition from SBOL

The SBOL document is RDF/XML. The device's `ComponentDefinition` lists its subparts as `Component`s, and each `SequenceAnnotation` places one subpart on the device sequence with 1-indexed, inclusive coordinates. Recursive SBOL also carries each subpart's own definition, so the parts can be named. The FASTA view keeps only the flattened sequence.

In [8]:
import xml.etree.ElementTree as ET

SBOL = "{http://sbols.org/v2#}"
DCTERMS = "{http://purl.org/dc/terms/}"
RDF = "{http://www.w3.org/1999/02/22-rdf-syntax-ns#}"

sbol = next(d for d in result.documents if d.format == "sbol")
root = ET.fromstring(sbol.content)
definitions = {d.get(f"{RDF}about"): d for d in root.iter(f"{SBOL}ComponentDefinition")}
device = next(d for d in definitions.values() if d.findtext(f"{SBOL}displayId") == "BBa_I13504")

# Map each Component in the device to the part it instantiates
parts = {
    c.get(f"{RDF}about"): definitions[c.find(f"{SBOL}definition").get(f"{RDF}resource")]
    for c in device.iter(f"{SBOL}Component")
}
print(f"{device.findtext(f'{SBOL}displayId')}: {device.findtext(f'{DCTERMS}description')}")
for annotation in device.iter(f"{SBOL}SequenceAnnotation"):
    part = parts[annotation.find(f"{SBOL}component").get(f"{RDF}resource")]
    start = int(annotation.findtext(f".//{SBOL}start"))
    end = int(annotation.findtext(f".//{SBOL}end"))
    print(f"  {start:>4}-{end:<4} {part.findtext(f'{SBOL}displayId'):<10} {part.findtext(f'{DCTERMS}description')}")

fasta = next(d for d in result.documents if d.format == "fasta")
header, *lines = fasta.content.splitlines()
print()
print(header)
print(f"{len(''.join(lines))} nt")

BBa_I13504: Screening plasmid intermediate
     1-12   BBa_B0034  RBS (Elowitz 1999) -- defines RBS efficiency
    19-738  BBa_E0040  green fluorescent protein derived from jellyfish Aequeora victoria wild-type GFP (SwissProt: P42212
   747-826  BBa_B0010  T1 from E. coli rrnB
   835-875  BBa_B0012  TE from coliphageT7

>BBa_I13504 sequence 1 (875 bp)
875 nt


#### Non-recursive SBOL

Set `recursive_sbol=False` to download only the top-level object, without the child objects it references. Recursive SBOL is the right choice for reconstructing a hierarchical design; the non-recursive document is smaller.

In [9]:
flat = run_synbiohub_fetch(inputs, SynBioHubFetchConfig(formats=["sbol"], recursive_sbol=False))
print(f"Recursive SBOL:     {len(sbol.content):>7,} chars")
print(f"Non-recursive SBOL: {len(flat.documents[0].content):>7,} chars  {flat.documents[0].source_url}")

Recursive SBOL:      38,793 chars
Non-recursive SBOL:  11,210 chars  https://api.synbiohub.org/public/igem/BBa_I13504/1/sbolnr


## Export Results

The output exports to JSON, with every document and its source URL.

In [10]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export("synbiohub_bba_i13504", export_path=output_dir, file_format="json")
print(f"Exported to {output_dir / 'synbiohub_bba_i13504.json'}")

Exported to example_output/synbiohub_bba_i13504.json
